# PyTorch spectral datasets and CUDA verification

This notebook validates `TorchPreprocessedDataset`, method-aware spectral collation, pinned host memory, non-blocking CUDA transfer, and minimal finite `Conv2d` forward/backward passes for FFT, Morlet, Superlet, and STFT.

Each method uses one canonical `Data_Train/exec` block and one canonical `Data_Pattern/patt` block through the existing spectral cache. This is an infrastructure smoke test, not model training or a scientific comparison.

In [1]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.datasets import (
    FFTDataset,
    MorletDataset,
    STFTDataset,
    SuperletDataset,
    TorchPreprocessedDataset,
    collate_torch_spectral_samples,
)

assert torch.cuda.is_available(), "This verification notebook requires CUDA"
DEVICE = torch.device("cuda")
METHOD_DATASETS = {
    "fft": FFTDataset,
    "morlet": MorletDataset,
    "superlet": SuperletDataset,
    "stft": STFTDataset,
}
{
    "torch_version": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "device": torch.cuda.get_device_name(DEVICE),
    "methods": tuple(METHOD_DATASETS),
}

{'torch_version': '2.12.0+cu130',
 'cuda_runtime': '13.0',
 'device': 'NVIDIA GeForce RTX 3070 Ti',
 'methods': ('fft', 'morlet', 'superlet', 'stft')}

## Canonical cached samples

The existing preprocessing datasets remain responsible for validated transforms and disk caches. The Torch adapter only exposes their arrays as CPU tensors.

In [2]:
method_samples = {}
source_shapes = {}
for method, dataset_type in METHOD_DATASETS.items():
    exec_dataset = TorchPreprocessedDataset(
        dataset_type(
            PROJECT_ROOT / "data" / "Data_Train",
            dataset_step_type="exec",
        )
    )
    patt_dataset = TorchPreprocessedDataset(
        dataset_type(
            PROJECT_ROOT / "data" / "Data_Pattern",
            dataset_step_type="patt",
        )
    )
    exec_sample = exec_dataset[0]
    patt_sample = patt_dataset[0]
    assert exec_sample.method == patt_sample.method == method
    assert exec_sample.eeg_power.device.type == patt_sample.eeg_power.device.type == "cpu"
    method_samples[method] = [exec_sample, patt_sample]
    source_shapes[method] = {
        "Data_Train/exec": tuple(exec_sample.eeg_power.shape),
        "Data_Pattern/patt": tuple(patt_sample.eeg_power.shape),
    }

source_shapes

{'fft': {'Data_Train/exec': (63, 39), 'Data_Pattern/patt': (63, 39)},
 'morlet': {'Data_Train/exec': (63, 39, 53),
  'Data_Pattern/patt': (63, 39, 92)},
 'superlet': {'Data_Train/exec': (63, 39, 50),
  'Data_Pattern/patt': (63, 39, 89)},
 'stft': {'Data_Train/exec': (63, 39, 55), 'Data_Pattern/patt': (63, 39, 94)}}

## Method-aware pinned batches

FFT has no spectral time metadata. Morlet, Superlet, and STFT pad their final time axis and keep independent spectral and original-EOG masks.

In [3]:
method_batches = {}
batch_summary = {}
for method, samples in method_samples.items():
    loader = DataLoader(
        samples,
        batch_size=2,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_torch_spectral_samples,
    )
    batch = next(iter(loader))
    assert batch.eeg_power.is_pinned()
    assert batch.eog.is_pinned() and batch.eog_finite_mask.is_pinned()
    assert batch.eog_time_mask.sum(dim=1).tolist() == batch.eog_lengths.tolist()
    if method == "fft":
        assert batch.eeg_power.ndim == 3
        assert batch.times is None
        assert batch.spectral_lengths is None
        assert batch.spectral_time_mask is None
    else:
        assert batch.eeg_power.ndim == 4
        assert batch.times is not None and batch.times.is_pinned()
        assert batch.spectral_lengths is not None and batch.spectral_lengths.is_pinned()
        assert batch.spectral_time_mask is not None and batch.spectral_time_mask.is_pinned()
        assert batch.spectral_time_mask.sum(dim=1).tolist() == batch.spectral_lengths.tolist()
        for index, length in enumerate(batch.spectral_lengths.tolist()):
            assert torch.count_nonzero(batch.eeg_power[index, :, :, length:]) == 0
            assert torch.count_nonzero(batch.times[index, length:]) == 0
    method_batches[method] = batch
    batch_summary[method] = {
        "power_shape": tuple(batch.eeg_power.shape),
        "spectral_lengths": None if batch.spectral_lengths is None else batch.spectral_lengths.tolist(),
        "eog_lengths": batch.eog_lengths.tolist(),
        "pinned": batch.eeg_power.is_pinned(),
    }

batch_summary

{'fft': {'power_shape': (2, 63, 39),
  'spectral_lengths': None,
  'eog_lengths': [16001, 26001],
  'pinned': True},
 'morlet': {'power_shape': (2, 63, 39, 92),
  'spectral_lengths': [53, 92],
  'eog_lengths': [16001, 26001],
  'pinned': True},
 'superlet': {'power_shape': (2, 63, 39, 89),
  'spectral_lengths': [50, 89],
  'eog_lengths': [16001, 26001],
  'pinned': True},
 'stft': {'power_shape': (2, 63, 39, 94),
  'spectral_lengths': [55, 94],
  'eog_lengths': [16001, 26001],
  'pinned': True}}

## CUDA forward/backward checks

A tiny independent `Conv2d` model is created for each representation. FFT is viewed as a one-channel channel-by-frequency image; TFR methods use EEG channels as input channels over frequency and time.

In [4]:
cuda_results = {}
for method, batch in method_batches.items():
    gpu_batch = batch.to(DEVICE, non_blocking=True)
    assert gpu_batch.eeg_power.is_cuda
    assert gpu_batch.frequencies.is_cuda
    assert gpu_batch.eog.is_cuda
    assert gpu_batch.samples is batch.samples

    model_input = gpu_batch.eeg_power.unsqueeze(1) if method == "fft" else gpu_batch.eeg_power
    model = torch.nn.Sequential(
        torch.nn.Conv2d(model_input.shape[1], 4, kernel_size=3, stride=2),
        torch.nn.GELU(),
        torch.nn.AdaptiveAvgPool2d(1),
        torch.nn.Flatten(),
        torch.nn.Linear(4, 1),
    ).to(DEVICE)

    prediction = model(model_input)
    loss = prediction.square().mean()
    loss.backward()
    torch.cuda.synchronize()

    assert torch.isfinite(prediction).all()
    assert torch.isfinite(loss)
    assert all(parameter.grad is not None for parameter in model.parameters())
    assert all(torch.isfinite(parameter.grad).all() for parameter in model.parameters())
    cuda_results[method] = {
        "prediction_shape": tuple(prediction.shape),
        "loss": float(loss.detach().cpu()),
        "all_gradients_finite": True,
    }

result = {
    "status": "CUDA_VERIFIED",
    "device": str(DEVICE),
    "methods": cuda_results,
}
result

{'status': 'CUDA_VERIFIED',
 'device': 'cuda',
 'methods': {'fft': {'prediction_shape': (2, 1),
   'loss': 0.0791022926568985,
   'all_gradients_finite': True},
  'morlet': {'prediction_shape': (2, 1),
   'loss': 0.01744397170841694,
   'all_gradients_finite': True},
  'superlet': {'prediction_shape': (2, 1),
   'loss': 0.18341012299060822,
   'all_gradients_finite': True},
  'stft': {'prediction_shape': (2, 1),
   'loss': 0.034211382269859314,
   'all_gradients_finite': True}}}

## Conclusion

All four cached spectral representations convert to typed CPU tensors, collate with the correct method-specific contract, transfer from pinned memory to CUDA, and complete finite `Conv2d` forward/backward checks.